# Flickr30k T3S3_LogitKD three-seed aggregate

Tổng hợp seeds `42, 123, 2026`, đối chiếu baseline cùng seed và thực hiện
paired bootstrap CIDEr-D trên đúng 1.014 ảnh Validation.

## Kaggle input bắt buộc — đúng 7 input

1. Output `F30K_S3_42`.
2. Output `F30K_S3_123`.
3. Output `F30K_S3_2026`.
4. Output `F30K_T3S3_LogitKD_42`.
5. Output `F30K_T3S3_LogitKD_123`.
6. Output `F30K_T3S3_LogitKD_2026`.
7. Package/dataset `pycocoevalcap-1.2` offline.

Runtime: CPU/Accelerator None, Internet Off, dự kiến 3–10 phút.
Không gắn Data, Flickr images, Teacher model hoặc Test.

## Cell 1 — Verify three seeds, checkpoints and Validation pairing

In [1]:
EXPERIMENT_ID = "F30K_T3S3_LogitKD_Aggregate"
PAIR_ID = "T3S3_LogitKD"
STUDENT_ID = "S3"
METHOD = "Direct Logit-KD"


import csv
import hashlib
import importlib.util
import json
import math
import os
import shutil
import statistics
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working") / EXPERIMENT_ID
WORK_ROOT.mkdir(parents=True, exist_ok=True)
SEEDS = (42, 123, 2026)
METRIC_NAMES = ("ce_loss", "BLEU_1", "BLEU_4", "METEOR", "ROUGE_L", "CIDEr_D")

def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

def unique_named(name):
    matches = sorted(path for path in INPUT_ROOT.rglob(name) if path.is_file())
    assert len(matches) == 1, (f"Cần đúng một {name}", [str(x) for x in matches])
    return matches[0]

if importlib.util.find_spec("pycocoevalcap") is None:
    wheels = sorted(INPUT_ROOT.rglob("pycocoevalcap*.whl"))
    if wheels:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", str(wheels[0])]
        )
    else:
        packages = [p for p in INPUT_ROOT.rglob("pycocoevalcap") if p.is_dir()]
        assert packages, "Hãy gắn input pycocoevalcap-1.2 offline."
        sys.path.insert(0, str(packages[0].parent))
assert importlib.util.find_spec("pycocoevalcap") is not None
assert shutil.which("java"), "METEOR runtime yêu cầu Java."

run_rows = []
caption_pairs = {}
for seed in SEEDS:
    baseline_id = f"F30K_{STUDENT_ID}_{seed}"
    kd_id = f"F30K_{PAIR_ID}_{seed}"
    baseline_result_path = unique_named(baseline_id + "_RESULT.json")
    kd_result_path = unique_named(kd_id + "_RESULT.json")
    baseline = read_json(baseline_result_path)
    kd = read_json(kd_result_path)

    assert baseline["status"] == "complete"
    assert baseline["experiment_id"] == baseline_id
    assert baseline["model_id"] == STUDENT_ID
    assert int(baseline["seed"]) == seed
    assert baseline["test_set_status"] == "sealed"
    assert baseline["test_set_used"] is False
    assert kd["status"] == "complete"
    assert kd["experiment_id"] == kd_id
    assert kd["student_id"] == STUDENT_ID
    assert int(kd["seed"]) == seed
    assert kd["method"] == METHOD
    assert kd["test_set_status"] == "sealed"
    assert kd["test_set_used"] is False

    baseline_best = baseline_result_path.parent / "best.pt"
    kd_best = kd_result_path.parent / "best.pt"
    assert sha256(baseline_best) == baseline["artifacts"]["best.pt"]["sha256"]
    assert sha256(kd_best) == kd["artifacts"]["best.pt"]["sha256"]

    baseline_captions_path = (
        baseline_result_path.parent / "best_validation_captions.json"
    )
    kd_captions_path = kd_result_path.parent / "best_validation_captions.json"
    assert sha256(baseline_captions_path) == (
        baseline["artifacts"]["best_validation_captions.json"]["sha256"]
    )
    assert sha256(kd_captions_path) == (
        kd["artifacts"]["best_validation_captions.json"]["sha256"]
    )
    baseline_captions = read_json(baseline_captions_path)
    kd_captions = read_json(kd_captions_path)
    assert baseline_captions["references"] == kd_captions["references"]
    assert len(baseline_captions["references"]) == 1014
    assert set(baseline_captions["hypotheses"]) == set(kd_captions["hypotheses"])
    caption_pairs[seed] = (baseline_captions, kd_captions)

    baseline_metrics = baseline["best_validation_metrics"]
    kd_metrics = kd["best_validation_metrics"]
    row = {
        "pair_id": PAIR_ID, "method": METHOD, "seed": seed,
        "baseline_best_epoch": int(baseline["best_epoch"]),
        "kd_best_epoch": int(kd["best_epoch"]),
    }
    for metric in METRIC_NAMES:
        row[f"baseline_{metric}"] = float(baseline_metrics[metric])
        row[f"kd_{metric}"] = float(kd_metrics[metric])
        if metric != "ce_loss":
            row[f"delta_{metric}"] = (
                float(kd_metrics[metric]) - float(baseline_metrics[metric])
            )
    reported_delta = float(kd["delta_validation_cider_d"])
    assert math.isclose(
        reported_delta, row["delta_CIDEr_D"], rel_tol=0.0, abs_tol=1e-12
    )
    run_rows.append(row)

print("Authorization          : PASSED")
print("Pair / method          :", PAIR_ID, "/", METHOD)
print("Runs verified          : 3 / 3")
print("Validation images      : 1014")
print("Held-out Test          : SEALED / NOT READ")

Authorization          : PASSED
Pair / method          : T3S3_LogitKD / Direct Logit-KD
Runs verified          : 3 / 3
Validation images      : 1014
Held-out Test          : SEALED / NOT READ


## Cell 2 — Per-image CIDEr-D and 10.000 paired bootstrap replicates

In [2]:
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from pycocoevalcap.cider.cider import Cider

def per_image_cider(bundle):
    image_ids = sorted(int(value) for value in bundle["references"])
    references = {
        image_id: [
            {"caption": caption}
            for caption in bundle["references"][str(image_id)]
        ]
        for image_id in image_ids
    }
    hypotheses = {
        image_id: [
            {"caption": bundle["hypotheses"][str(image_id)][0]}
        ]
        for image_id in image_ids
    }
    tokenizer = PTBTokenizer()
    references = tokenizer.tokenize(references)
    hypotheses = tokenizer.tokenize(hypotheses)
    corpus_score, scores = Cider().compute_score(references, hypotheses)
    scores = np.asarray(scores, dtype=np.float64)
    assert scores.shape == (1014,)
    assert math.isclose(float(scores.mean()), float(corpus_score), abs_tol=1e-12)
    return image_ids, scores

bootstrap_rows = []
all_seed_differences = []
for seed in SEEDS:
    baseline_bundle, kd_bundle = caption_pairs[seed]
    baseline_ids, baseline_scores = per_image_cider(baseline_bundle)
    kd_ids, kd_scores = per_image_cider(kd_bundle)
    assert baseline_ids == kd_ids
    difference = kd_scores - baseline_scores
    all_seed_differences.append(difference)
    rng = np.random.default_rng(20261026 + seed)
    replicate_means = np.empty(10000, dtype=np.float64)
    for start in range(0, 10000, 250):
        stop = min(10000, start + 250)
        indices = rng.integers(
            0, len(difference), size=(stop - start, len(difference))
        )
        replicate_means[start:stop] = difference[indices].mean(axis=1)
    lower, upper = np.quantile(replicate_means, [0.025, 0.975])
    observed = float(difference.mean())
    positive_probability = float((replicate_means > 0).mean())
    bootstrap_rows.append({
        "pair_id": PAIR_ID, "seed": seed, "images": len(difference),
        "observed_delta_CIDEr_D": observed,
        "ci95_lower": float(lower), "ci95_upper": float(upper),
        "bootstrap_positive_probability": positive_probability,
        "replicates": 10000,
    })

stacked_differences = np.stack(all_seed_differences, axis=0)
assert stacked_differences.shape == (3, 1014)
rng = np.random.default_rng(20261026)
combined_replicates = np.empty(10000, dtype=np.float64)
for start in range(0, 10000, 250):
    stop = min(10000, start + 250)
    indices = rng.integers(
        0, stacked_differences.shape[1],
        size=(stop - start, stacked_differences.shape[1]),
    )
    # Cùng một tập 1.014 ảnh được dùng ở cả ba seed. Lấy mẫu ảnh một lần
    # rồi trung bình qua seed để giữ đúng cấu trúc paired/hierarchical.
    combined_replicates[start:stop] = (
        stacked_differences[:, indices].mean(axis=(0, 2))
    )
combined_lower, combined_upper = np.quantile(
    combined_replicates, [0.025, 0.975]
)
combined_bootstrap = {
    "unique_validation_images": 1014,
    "seed_count": 3,
    "resampling_unit": "image_id shared across seeds",
    "observed_delta_CIDEr_D": float(stacked_differences.mean()),
    "ci95_lower": float(combined_lower),
    "ci95_upper": float(combined_upper),
    "bootstrap_positive_probability": float(
        (combined_replicates > 0).mean()
    ),
    "replicates": 10000,
}
print("Paired bootstrap       : COMPLETE")
for row in bootstrap_rows:
    print(
        f"seed={row['seed']} delta={row['observed_delta_CIDEr_D']:+.6f} "
        f"CI95=[{row['ci95_lower']:+.6f}, {row['ci95_upper']:+.6f}]"
    )

PTBTokenizer tokenized 74030 tokens at 244176.81 tokens per second.
PTBTokenizer tokenized 13691 tokens at 99243.04 tokens per second.
PTBTokenizer tokenized 74030 tokens at 264245.59 tokens per second.
PTBTokenizer tokenized 13417 tokens at 90516.99 tokens per second.
PTBTokenizer tokenized 74030 tokens at 266644.62 tokens per second.
PTBTokenizer tokenized 13330 tokens at 100160.90 tokens per second.
PTBTokenizer tokenized 74030 tokens at 281199.15 tokens per second.
PTBTokenizer tokenized 13705 tokens at 96052.76 tokens per second.
PTBTokenizer tokenized 74030 tokens at 318421.45 tokens per second.
PTBTokenizer tokenized 13598 tokens at 98369.34 tokens per second.
PTBTokenizer tokenized 74030 tokens at 263995.75 tokens per second.
PTBTokenizer tokenized 13530 tokens at 94299.39 tokens per second.


Paired bootstrap       : COMPLETE
seed=42 delta=+0.046364 CI95=[+0.025177, +0.068002]
seed=123 delta=+0.022454 CI95=[-0.001564, +0.046337]
seed=2026 delta=+0.032859 CI95=[+0.010039, +0.055931]


## Cell 3 — Aggregate statistics, decision gate and manifest

In [3]:
summary_metrics = {}
for metric in METRIC_NAMES:
    baseline_values = [row[f"baseline_{metric}"] for row in run_rows]
    kd_values = [row[f"kd_{metric}"] for row in run_rows]
    metric_summary = {
        "baseline_mean": statistics.mean(baseline_values),
        "baseline_sample_sd": statistics.stdev(baseline_values),
        "kd_mean": statistics.mean(kd_values),
        "kd_sample_sd": statistics.stdev(kd_values),
    }
    if metric != "ce_loss":
        deltas = [row[f"delta_{metric}"] for row in run_rows]
        metric_summary["delta_mean"] = statistics.mean(deltas)
        metric_summary["delta_sample_sd"] = statistics.stdev(deltas)
    summary_metrics[metric] = metric_summary

cider_deltas = [row["delta_CIDEr_D"] for row in run_rows]
consistent_positive = all(value > 0 for value in cider_deltas)
consistent_negative = all(value < 0 for value in cider_deltas)
if consistent_positive:
    decision = "PROMOTE_TO_FINAL_METHOD_LOCK"
    next_phase = f"F30K_{PAIR_ID}_Final_Lock"
elif consistent_negative:
    decision = "STOP_VALIDATION_NO_GAIN"
    next_phase = None
else:
    decision = "INCONCLUSIVE_MIXED_SEED_DIRECTION"
    next_phase = None

result = {
    "schema_version": "f30k_kd_three_seed_aggregate_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "complete", "experiment_id": EXPERIMENT_ID,
    "pair_id": PAIR_ID, "method": METHOD,
    "seeds": list(SEEDS), "run_count": 3,
    "validation_images_per_run": 1014,
    "selection_metric": "Validation CIDEr-D",
    "runs": run_rows, "summary": summary_metrics,
    "paired_bootstrap": bootstrap_rows,
    "combined_paired_bootstrap": combined_bootstrap,
    "consistent_positive_delta": consistent_positive,
    "consistent_negative_delta": consistent_negative,
    "decision": decision, "next_phase_authorized": next_phase,
    "test_set_status": "sealed", "test_set_used": False,
}

result_path = WORK_ROOT / f"{EXPERIMENT_ID}_RESULT.json"
run_csv = WORK_ROOT / "three_seed_metrics.csv"
bootstrap_csv = WORK_ROOT / "paired_bootstrap_cider.csv"
report_path = WORK_ROOT / "REPORT.md"
write_json(result_path, result)

with run_csv.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(run_rows[0]))
    writer.writeheader()
    writer.writerows(run_rows)
with bootstrap_csv.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(bootstrap_rows[0]))
    writer.writeheader()
    writer.writerows(bootstrap_rows)

lines = [
    f"# {PAIR_ID} — three-seed aggregate",
    "",
    f"- Method: `{METHOD}`",
    f"- Decision: `{decision}`",
    f"- Mean CIDEr-D delta: `{summary_metrics['CIDEr_D']['delta_mean']:+.6f}`",
    f"- Combined bootstrap CI95: "
    f"`[{combined_bootstrap['ci95_lower']:+.6f}; "
    f"{combined_bootstrap['ci95_upper']:+.6f}]`",
    "- Test: `SEALED / NOT USED`",
    "",
    "| Seed | Baseline CIDEr-D | KD CIDEr-D | Delta | CI95 |",
    "|---:|---:|---:|---:|---:|",
]
bootstrap_by_seed = {row["seed"]: row for row in bootstrap_rows}
for row in run_rows:
    boot = bootstrap_by_seed[row["seed"]]
    lines.append(
        f"| {row['seed']} | {row['baseline_CIDEr_D']:.6f} | "
        f"{row['kd_CIDEr_D']:.6f} | {row['delta_CIDEr_D']:+.6f} | "
        f"[{boot['ci95_lower']:+.6f}; {boot['ci95_upper']:+.6f}] |"
    )
report_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

manifest_path = WORK_ROOT / "MANIFEST.json"
artifacts = []
for path in [result_path, run_csv, bootstrap_csv, report_path]:
    artifacts.append({
        "relative_path": path.name, "size_bytes": path.stat().st_size,
        "sha256": sha256(path),
    })
write_json(manifest_path, {
    "schema_version": "f30k_kd_aggregate_manifest_v1",
    "experiment_id": EXPERIMENT_ID, "status": "complete",
    "artifacts": artifacts,
    "test_set_status": "sealed", "test_set_used": False,
})

print("Status                 : COMPLETE")
print("Decision               :", decision)
print("Mean CIDEr-D delta     :", summary_metrics["CIDEr_D"]["delta_mean"])
print("Combined CI95          :", combined_bootstrap["ci95_lower"],
      combined_bootstrap["ci95_upper"])
print("Next                   :", next_phase)
print("Held-out Test          : SEALED / NOT USED")

Status                 : COMPLETE
Decision               : PROMOTE_TO_FINAL_METHOD_LOCK
Mean CIDEr-D delta     : 0.03389222163311526
Combined CI95          : 0.019477099305418074 0.04817347479190771
Next                   : F30K_T3S3_LogitKD_Final_Lock
Held-out Test          : SEALED / NOT USED
